# Week 06A — Deep Learning I: From Hand-Crafted to Learned Features

**MCTA 4364 Machine Vision** · Session 1 of Week 6 · 80 minutes

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hasanzaki/MCTA-4364-Machine-Vision/blob/main/notebooks/W06_deep_learning/W06A_learned_features.ipynb) [![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/hasanzaki/MCTA-4364-Machine-Vision/blob/main/notebooks/W06_deep_learning/W06A_learned_features.ipynb)

In Weeks 3–5 **you** designed the features: Sobel kernels, thresholds, HOG and SIFT descriptors.
This week we let the computer **learn** them from examples. By the end of this session you will have
built, trained and dissected a convolutional neural network (CNN) and measured it against the
hand-crafted pipeline you already know.

### Learning outcomes
By the end of this session you will be able to:
1. Explain why hand-crafted features struggle with **intra-class variation**.
2. Describe an artificial **neuron**, common **activation functions** and **gradient descent**, with the maths.
3. Show that a CNN convolution layer is exactly the `cv2.filter2D` operation from Week 3, but with **learned** kernels.
4. Compute a convolution layer's **output size**, **parameter count** and **receptive field**.
5. Train a CNN in PyTorch and compare it quantitatively with a **HOG + linear classifier** baseline.
6. Visualise what a CNN has learned: filters, feature maps and embeddings.

### Session plan (80 min)
| Time | Section | Activity |
|---|---|---|
| 0–10 | 1–3 | Where hand-crafted features break; HOG baseline |
| 10–25 | 4–5 | Neurons, activations, gradient descent (interactive) |
| 25–35 | 6–7 | Multi-layer networks; why convolution for images |
| 35–45 | 8 | CNN building blocks: output size, pooling, receptive field |
| 45–65 | 9–10 | Build and train a CNN; look inside it |
| 65–75 | 11 | Results and error analysis |
| 75–80 | 14 | Quiz and exit ticket |
| Home | 12–13 | Exercises with self-checks, challenge |

**Prerequisites:** Week 3B (convolution), Week 5 (HOG/SIFT descriptors), basic Python/NumPy.

**Hardware:** runs on a laptop with a 4 GB NVIDIA GPU in about 5 minutes, or on a CPU with smaller settings. Also runs in Colab/Kaggle.

## 0. Setup

In [ ]:
# Setup: works in Google Colab, Kaggle, Jupyter/VS Code on your laptop, and CI.
import importlib.util
import os
import subprocess
import sys

REPO_URL = "https://github.com/hasanzaki/MCTA-4364-Machine-Vision.git"
if not os.path.exists("resources/scripts/cvhelpers.py"):
    if os.path.exists("../../resources/scripts/cvhelpers.py"):  # opened from its own folder
        os.chdir("../..")
    else:  # Colab / Kaggle: fetch the course helpers
        if not os.path.isdir("MCTA-4364-Machine-Vision"):
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
        os.chdir("MCTA-4364-Machine-Vision")
print("Working directory:", os.getcwd())

# Install only what is missing. PyTorch is deliberately NOT pip-installed here:
# that would replace your laptop's CUDA build with a CPU-only one (see SETUP.md).
needed = {"cv2": "opencv-python", "skimage": "scikit-image", "ipywidgets": "ipywidgets",
          "matplotlib": "matplotlib", "scipy": "scipy"}
missing = [pkg for mod, pkg in needed.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
if importlib.util.find_spec("torch") is None:
    raise ImportError("PyTorch is missing. Install it with CUDA support as described in "
                      "SETUP.md, or use Google Colab where it is preinstalled.")
sys.path.append("resources/scripts")

In [ ]:
import time

import cv2
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import scipy.ndimage as ndi
import skimage.data
import torch
import torch.nn.functional as F
from skimage.feature import hog
from skimage.filters import gabor_kernel
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets

from cvhelpers import SMOKE, check, data_dir, interact, pipeline_diagram, quiz, youtube
from dlhelpers import (confusion_matrix, count_params, hw_profile, layer_table,
                       plot_confusion, plot_history, set_seed)

plt.rcParams.update({"figure.dpi": 100, "axes.titlesize": 11})
HW = hw_profile()
DEVICE = HW["device"]
set_seed(0)

# Sizes adapt to your hardware (CI uses the tiny SMOKE setting).
N_TRAIN = 5000 if SMOKE else (60000 if HW["cuda"] else 20000)
N_HOG = 2000 if SMOKE else 10000
EPOCHS = 2 if SMOKE else (8 if HW["cuda"] else 3)
BATCH = 128
print(f"Training images: {N_TRAIN} | epochs: {EPOCHS} | batch: {BATCH}")

In [ ]:
stages = ["W1-2\nImages &\ncameras", "W3\nEnhance &\nfilter", "W4\nSegment", "W5\nFeatures &\nmatching",
          "W6\nLEARNED\nfeatures", "W7\nIntegration\n& midterm", "W8-13\nDetect, track,\nsegment, 3D, VLM"]
colors = ["#e8f0fe"] * 4 + ["#fce8b2"] + ["#f1f3f4"] * 2
pipeline_diagram(stages, title="Where this week fits in the course", colors=colors, figsize=(17, 2.3))

## 1. The story so far: who designs the features?

Every recognition pipeline you have built so far has the same shape: **image → features → decision**.
The difference between classical machine vision and deep learning is *who designs each stage*.

- **Weeks 3–5 (hand-crafted):** a human chooses the filters (Sobel, Gabor), the descriptor (HOG, SIFT)
  and often the decision rule (thresholds). This works when the designer can anticipate every variation: fixed lighting,
  a rigid part, a known orientation. That is why classical methods still dominate simple, controlled inspection.
- **This week (learned):** we choose only the *architecture* and supply *labelled examples*. The filters, descriptors
  and classifier are all optimised together to minimise the error on those examples.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(15, 4.6))
pipeline_diagram(["Image", "Designed filters\n(Sobel, Gabor)", "Designed descriptor\n(HOG, SIFT)",
                  "Classifier", "Label"], ax=axes[0], title="Weeks 3-5: hand-crafted pipeline",
                 colors=["#f1f3f4", "#fad2cf", "#fad2cf", "#ceead6", "#f1f3f4"],
                 notes=["", "chosen by a human", "chosen by a human", "learned from data", ""])
pipeline_diagram(["Image", "Conv layer 1\n(edge-like)", "Conv layers 2..n\n(textures, parts)",
                  "Classifier", "Label"], ax=axes[1], title="This week: every stage is learned (a CNN)",
                 colors=["#f1f3f4", "#ceead6", "#ceead6", "#ceead6", "#f1f3f4"],
                 notes=["", "learned from data", "learned from data", "learned from data", ""])
plt.tight_layout()
plt.show()

## 2. A task where hand-crafted features struggle

We use **Fashion-MNIST** (Xiao et al., 2017): 70,000 greyscale 28×28 images of 10 clothing categories, released
under the MIT licence as a harder drop-in replacement for handwritten digits. It is small enough to train on a
laptop in minutes, yet it has the key difficulty of real inspection data: **large variation within each class**
(shirts come in many shapes, patterns and poses) and **small differences between some classes** (shirt vs T-shirt vs pullover).

The first run downloads about 30 MB into the git-ignored `data/` folder.

In [ ]:
root = data_dir()
train_ds = datasets.FashionMNIST(root, train=True, download=True)
test_ds = datasets.FashionMNIST(root, train=False, download=True)
CLASSES = train_ds.classes

rng = np.random.default_rng(0)
idx = rng.permutation(len(train_ds.data))[:N_TRAIN]
Xtr, ytr = train_ds.data.numpy()[idx], train_ds.targets.numpy()[idx]
Xte, yte = test_ds.data.numpy(), test_ds.targets.numpy()
if SMOKE:
    Xte, yte = Xte[:2000], yte[:2000]
print("train:", Xtr.shape, "test:", Xte.shape, "| classes:", CLASSES)

**Figure 2.1 — Intra-class variation.** Each row is one class. Look how different the items in one row are.

In [ ]:
fig, axes = plt.subplots(10, 10, figsize=(11, 11))
for c in range(10):
    examples = np.where(ytr == c)[0][:10]
    for j, i in enumerate(examples):
        ax = axes[c, j]
        ax.imshow(Xtr[i], cmap="gray_r")
        ax.set_xticks([]); ax.set_yticks([])
        if j == 0:
            ax.set_ylabel(CLASSES[c], rotation=0, ha="right", va="center", fontsize=10)
plt.suptitle("Fashion-MNIST: 10 examples per class", fontweight="bold")
plt.tight_layout()
plt.show()

**Figure 2.2 — Why a single template fails.** A Week-4-style *template matcher* compares an image with the average image of
each class. The class means (top) are blurry, and the per-pixel standard deviation (bottom) shows how much each class varies.
Classes such as *Shirt*, *Pullover* and *Coat* have almost identical templates.

In [ ]:
means = np.stack([Xtr[ytr == c].mean(0) for c in range(10)])
stds = np.stack([Xtr[ytr == c].std(0) for c in range(10)])
fig, axes = plt.subplots(2, 10, figsize=(16, 3.6))
for c in range(10):
    axes[0, c].imshow(means[c], cmap="gray_r"); axes[0, c].set_title(CLASSES[c], fontsize=9)
    axes[1, c].imshow(stds[c], cmap="magma")
for ax in axes.ravel():
    ax.axis("off")
axes[0, 0].text(-8, 14, "mean", rotation=90, va="center")
axes[1, 0].text(-8, 14, "std", rotation=90, va="center")
plt.suptitle("Class templates (mean) and variability (standard deviation)", fontweight="bold")
plt.tight_layout()
plt.show()

# Nearest-template classifier: predict the class whose mean image is closest.
d = ((Xte[:, None].astype(np.float32) - means[None]) ** 2).sum((2, 3))
acc_template = (d.argmin(1) == yte).mean()
print(f"Nearest-template accuracy: {acc_template:.3f}")

## 3. A strong hand-crafted baseline: HOG + linear classifier

The **Histogram of Oriented Gradients** (Dalal & Triggs, 2005) is the hand-crafted descriptor that dominated pedestrian
detection before deep learning. It works like the SIFT descriptor from Week 5:
1. compute image gradients (Sobel, Week 3),
2. build a histogram of gradient **orientations** in each small cell (here 7×7 pixels, 9 bins),
3. normalise histograms over overlapping 2×2 blocks, which gives robustness to lighting.

A **linear classifier** then learns one weight per HOG feature per class: $\text{score}_c = \mathbf{w}_c^\top \mathbf{h} + b_c$.
The descriptor is hand-crafted, but the classifier is already learned from data.

In [ ]:
fig, axes = plt.subplots(2, 6, figsize=(14, 5))
for j, c in enumerate([0, 1, 5, 7, 8, 9]):
    img = Xtr[np.where(ytr == c)[0][0]]
    _, vis = hog(img, orientations=9, pixels_per_cell=(7, 7), cells_per_block=(2, 2), visualize=True)
    axes[0, j].imshow(img, cmap="gray_r"); axes[0, j].set_title(CLASSES[c])
    axes[1, j].imshow(vis, cmap="inferno"); axes[1, j].set_title("HOG cells")
for ax in axes.ravel():
    ax.axis("off")
plt.suptitle("Figure 3.1 - HOG turns each 7x7 cell into a histogram of edge directions", fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
def hog_features(images):
    return np.stack([hog(im, orientations=9, pixels_per_cell=(7, 7), cells_per_block=(2, 2))
                     for im in images]).astype(np.float32)

t0 = time.time()
H_tr, H_te = hog_features(Xtr[:N_HOG]), hog_features(Xte)
print(f"HOG features: {H_tr.shape[1]} numbers per image ({time.time() - t0:.1f}s)")


def train_linear(F_tr, y_tr, F_te, y_te, steps=300, lr=0.05):
    """Softmax (multinomial logistic) regression on fixed features, trained full-batch."""
    F_tr, F_te = torch.tensor(F_tr), torch.tensor(F_te)
    mu, sd = F_tr.mean(0), F_tr.std(0) + 1e-6
    F_tr, F_te = (F_tr - mu) / sd, (F_te - mu) / sd
    y_tr_t = torch.tensor(y_tr)
    clf = nn.Linear(F_tr.shape[1], 10)
    opt = torch.optim.Adam(clf.parameters(), lr=lr, weight_decay=1e-4)
    for _ in range(steps):
        loss = F.cross_entropy(clf(F_tr), y_tr_t)
        opt.zero_grad(); loss.backward(); opt.step()
    with torch.no_grad():
        pred = clf(F_te).argmax(1).numpy()
    return (pred == y_te).mean(), pred, count_params(clf)

acc_hog, pred_hog, params_hog = train_linear(H_tr, ytr[:N_HOG], H_te, yte)
print(f"HOG + linear classifier accuracy: {acc_hog:.3f}  ({params_hog:,} learned parameters)")
RESULTS = {"Nearest template": (acc_template, 7840), "HOG + linear": (acc_hog, params_hog)}

HOG is a big step up from templates, but its errors are concentrated in exactly the classes that differ in subtle
*shape details* (collar, sleeves, texture), which the designer did not encode. Keep this number in mind.

## 4. The artificial neuron

A neuron computes a weighted sum of its inputs, adds a bias and passes the result through a non-linear
**activation function** $\sigma$:

$$ y = \sigma\!\left(\sum_{i=1}^{n} w_i x_i + b\right) = \sigma(\mathbf{w}^\top\mathbf{x} + b) $$

- $\mathbf{w}$ (weights) and $b$ (bias) are **learned**.
- Without $\sigma$, stacking neurons would just give another linear function. The non-linearity is what lets
  networks represent complex decision boundaries.
- The linear classifier in Section 3 is exactly a layer of 10 such neurons (with a softmax on top).

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.axis("off"); ax.set_xlim(0, 10); ax.set_ylim(0, 5)
for k, yy in enumerate([4.2, 2.5, 0.8]):
    ax.add_patch(plt.Circle((1, yy), 0.35, fc="#e8f0fe", ec="#1a73e8"))
    ax.text(1, yy, f"$x_{k + 1}$", ha="center", va="center", fontsize=13)
    ax.annotate("", xy=(4.55, 2.5), xytext=(1.35, yy), arrowprops=dict(arrowstyle="-|>", color="#5f6368"))
    ax.text(2.8, (yy + 2.5) / 2 + 0.25, f"$w_{k + 1}$", fontsize=13, color="#1a73e8")
ax.add_patch(plt.Circle((5, 2.5), 0.5, fc="#fce8b2", ec="#e37400"))
ax.text(5, 2.5, "$\\Sigma$", ha="center", va="center", fontsize=18)
ax.text(5, 3.4, "$+b$", ha="center", fontsize=13)
ax.annotate("", xy=(6.6, 2.5), xytext=(5.5, 2.5), arrowprops=dict(arrowstyle="-|>", color="#5f6368"))
ax.add_patch(plt.Rectangle((6.6, 2.0), 1.3, 1.0, fc="#ceead6", ec="#188038"))
ax.text(7.25, 2.5, "$\\sigma(\\cdot)$", ha="center", va="center", fontsize=14)
ax.annotate("", xy=(9.2, 2.5), xytext=(7.9, 2.5), arrowprops=dict(arrowstyle="-|>", color="#5f6368"))
ax.text(9.4, 2.5, "$y$", va="center", fontsize=15)
ax.set_title("Figure 4.1 - One artificial neuron: weighted sum, bias, activation", fontweight="bold")
plt.show()

**Figure 4.2 — Activation functions (top) and their derivatives (bottom).** The derivative controls how much gradient flows backwards during learning. Sigmoid and tanh *saturate*: their gradient vanishes for large |z|. ReLU keeps a gradient of 1 for every positive input, which is a major reason deep networks became trainable (Krizhevsky et al., 2012).

In [ ]:
z = torch.linspace(-5, 5, 400, requires_grad=True)
acts = {"Sigmoid": torch.sigmoid, "Tanh": torch.tanh, "ReLU": torch.relu, "GELU": F.gelu}
fig, axes = plt.subplots(2, 4, figsize=(15, 5.5), sharex=True)
for j, (name, f) in enumerate(acts.items()):
    y = f(z)
    (grad,) = torch.autograd.grad(y.sum(), z)
    axes[0, j].plot(z.detach(), y.detach(), lw=2.5, color="#1a73e8"); axes[0, j].set_title(name)
    axes[1, j].plot(z.detach(), grad, lw=2.5, color="#e37400"); axes[1, j].set_title(f"d {name} / dz")
    for ax in axes[:, j]:
        ax.axhline(0, color="k", lw=0.6); ax.axvline(0, color="k", lw=0.6); ax.grid(alpha=0.3)
axes[1, 0].set_xlabel("z"); plt.tight_layout(); plt.show()

### Interactive: a single neuron as a classifier

The two point clouds below are two classes. A single sigmoid neuron draws a straight **decision boundary**
(where $w_1x_1 + w_2x_2 + b = 0$) and outputs a probability on either side. Move the sliders to separate the classes by hand.
In Section 5 we let gradient descent do it for us.

In [ ]:
rng = np.random.default_rng(1)
blob_a = rng.normal([-1.0, -1.0], 0.75, (120, 2))
blob_b = rng.normal([1.2, 1.0], 0.75, (120, 2))
P2 = np.vstack([blob_a, blob_b]); L2 = np.r_[np.zeros(120), np.ones(120)]
gx, gy = np.meshgrid(np.linspace(-4, 4, 200), np.linspace(-4, 4, 200))


def draw_neuron(ax, w1, w2, b):
    prob = 1 / (1 + np.exp(-(w1 * gx + w2 * gy + b)))
    ax.contourf(gx, gy, prob, levels=20, cmap="RdBu_r", alpha=0.55, vmin=0, vmax=1)
    ax.contour(gx, gy, prob, levels=[0.5], colors="k", linewidths=2)
    ax.scatter(*blob_a.T, c="#1a73e8", s=12, edgecolor="w", lw=0.3)
    ax.scatter(*blob_b.T, c="#d93025", s=12, edgecolor="w", lw=0.3)
    acc = (((w1 * P2[:, 0] + w2 * P2[:, 1] + b) > 0) == L2).mean()
    ax.set_title(f"w=({w1:.1f}, {w2:.1f}), b={b:.1f}  ->  accuracy {acc:.2f}")
    ax.set_xlim(-4, 4); ax.set_ylim(-4, 4); ax.set_aspect("equal")


fig, axes = plt.subplots(1, 3, figsize=(15, 4.8))
for ax, params in zip(axes, [(1.0, -1.0, 0.0), (0.2, 1.5, 2.0), (1.5, 1.5, 0.0)]):
    draw_neuron(ax, *params)
plt.suptitle("Figure 4.3 - Three neurons: poor, shifted, and good decision boundaries", fontweight="bold")
plt.tight_layout(); plt.show()


def neuron_demo(w1=1.0, w2=-1.0, b=0.0):
    fig, ax = plt.subplots(figsize=(5.5, 5.5))
    draw_neuron(ax, w1, w2, b)
    plt.show()


interact(neuron_demo, w1=widgets.FloatSlider(value=1.0, min=-3, max=3, step=0.1),
         w2=widgets.FloatSlider(value=-1.0, min=-3, max=3, step=0.1),
         b=widgets.FloatSlider(value=0.0, min=-4, max=4, step=0.1))

## 5. Learning = minimising a loss with gradient descent

For classification we measure the error with the **cross-entropy loss**. The network outputs scores $z_c$, and
softmax turns them into probabilities $p_c = e^{z_c} / \sum_k e^{z_k}$. For the true class $y$:

$$ \mathcal{L} = -\log p_y $$

The loss is 0 when the network puts all probability on the correct class, and grows quickly when it is confidently wrong.

**Gradient descent** repeatedly nudges every weight a small step *downhill*:

$$ \mathbf{w} \leftarrow \mathbf{w} - \eta \, \nabla_{\mathbf{w}} \mathcal{L} $$

- $\eta$ is the **learning rate**. Too small and learning crawls; too large and it overshoots and **diverges**.
- In practice the gradient is estimated on a **mini-batch** of examples (e.g. 128): this is *stochastic* gradient descent (SGD).
- **Momentum** keeps a running average of past gradients, which smooths the zig-zag in narrow valleys.
- **Backpropagation** (Rumelhart, Hinton & Williams, 1986) computes $\nabla_{\mathbf{w}}\mathcal{L}$ for millions of weights
  efficiently using the chain rule. PyTorch does this automatically with `loss.backward()`.

**Figure 5.1** uses a toy loss $\mathcal{L}(w_1, w_2) = \tfrac12(w_1^2 + 10\,w_2^2)$, a long narrow valley like many real loss surfaces.

In [ ]:
def toy_loss(w):
    return 0.5 * (w[..., 0] ** 2 + 10 * w[..., 1] ** 2)


def toy_grad(w):
    return np.array([w[0], 10 * w[1]])


def descend(lr, steps=30, momentum=0.0, start=(-4.5, 1.8)):
    w, v, path = np.array(start, float), np.zeros(2), [np.array(start, float)]
    for _ in range(steps):
        v = momentum * v - lr * toy_grad(w)
        w = w + v
        path.append(w.copy())
    return np.array(path)


W1, W2 = np.meshgrid(np.linspace(-5, 5, 200), np.linspace(-2.5, 2.5, 200))
Z = toy_loss(np.stack([W1, W2], -1))
runs = {"lr = 0.02 (too small)": (0.02, 0.0), "lr = 0.15 (good)": (0.15, 0.0),
        "lr = 0.205 (too large)": (0.205, 0.0), "lr = 0.05 + momentum 0.8": (0.05, 0.8)}
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for ax, (name, (lr, mom)), color in zip(axes, runs.items(), ["#9aa0a6", "#188038", "#d93025", "#1a73e8"]):
    path = descend(lr, momentum=mom)
    ax.contour(W1, W2, Z, levels=np.logspace(-1, 2.2, 14), cmap="viridis", linewidths=0.8)
    ax.plot(path[:, 0], path[:, 1], "o-", color=color, ms=3, lw=1.5)
    ax.plot(0, 0, "k*", ms=12); ax.set_title(name); ax.set_ylim(-2.5, 2.5)
    axes[4].semilogy(toy_loss(path) + 1e-8, color=color, lw=2, label=name)
axes[4].set_title("Loss vs step"); axes[4].set_xlabel("step"); axes[4].legend(fontsize=7); axes[4].grid(alpha=0.3)
plt.suptitle("Figure 5.1 - Gradient descent on a narrow valley", fontweight="bold")
plt.tight_layout(); plt.show()

### Interactive: tune the learning rate and momentum yourself

In [ ]:
def gd_demo(learning_rate=0.1, momentum=0.0, steps=30):
    path = descend(learning_rate, steps, momentum)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].contour(W1, W2, Z, levels=np.logspace(-1, 2.2, 14), cmap="viridis", linewidths=0.8)
    axes[0].plot(np.clip(path[:, 0], -5, 5), np.clip(path[:, 1], -2.5, 2.5), "o-", color="#d93025", ms=3)
    axes[0].plot(0, 0, "k*", ms=12); axes[0].set_title("Path on the loss surface")
    axes[1].semilogy(toy_loss(path) + 1e-8, lw=2); axes[1].grid(alpha=0.3)
    axes[1].set_title(f"final loss = {toy_loss(path[-1]):.3g}"); axes[1].set_xlabel("step")
    plt.tight_layout(); plt.show()


interact(gd_demo, learning_rate=widgets.FloatLogSlider(value=0.1, base=10, min=-2.5, max=-0.6, step=0.05),
         momentum=widgets.FloatSlider(value=0.0, min=0, max=0.95, step=0.05),
         steps=widgets.IntSlider(value=30, min=5, max=100, step=5))

### Autograd check: the chain rule by hand vs PyTorch

For one neuron $y = \text{ReLU}(wx + b)$ with loss $\mathcal{L} = (y - t)^2$, the chain rule gives
$\frac{\partial \mathcal{L}}{\partial w} = 2(y - t)\cdot \mathbb{1}[wx+b>0]\cdot x$. PyTorch's autograd computes the same number.

In [ ]:
x_val, t_val = 2.0, 1.0
w = torch.tensor(0.8, requires_grad=True)
b = torch.tensor(0.1, requires_grad=True)
y = torch.relu(w * x_val + b)
loss = (y - t_val) ** 2
loss.backward()
manual = 2 * (y.item() - t_val) * (1.0 if w.item() * x_val + b.item() > 0 else 0.0) * x_val
print(f"y = {y.item():.3f}, loss = {loss.item():.4f}")
print(f"dL/dw by hand: {manual:.4f} | autograd: {w.grad.item():.4f}")

Grant Sanderson (3Blue1Brown) gives an excellent visual explanation of gradient descent:

In [ ]:
youtube("IHZwWFHWa-w")

## 6. Stacking neurons: the multi-layer perceptron (MLP)

One neuron can only draw a straight line. Stacking layers of neurons, each followed by a non-linearity, lets a network
bend the boundary into any shape. The *universal approximation theorem* says one wide-enough hidden layer can approximate
any continuous function. Depth makes this far more efficient.

$$ \mathbf{h}_1 = \text{ReLU}(W_1\mathbf{x} + \mathbf{b}_1),\quad \mathbf{h}_2 = \text{ReLU}(W_2\mathbf{h}_1 + \mathbf{b}_2),\quad \mathbf{z} = W_3\mathbf{h}_2 + \mathbf{b}_3 $$

**Figure 6.1** trains a small MLP on the "two moons" dataset, which no straight line can separate, and shows how the boundary develops.

In [ ]:
def make_moons(n=400, noise=0.15, seed=0):
    r = np.random.default_rng(seed)
    t = r.uniform(0, np.pi, n // 2)
    upper = np.c_[np.cos(t), np.sin(t)]
    lower = np.c_[1 - np.cos(t), 0.5 - np.sin(t)]
    X = np.vstack([upper, lower]) + r.normal(0, noise, (n, 2))
    return X.astype(np.float32), np.r_[np.zeros(n // 2), np.ones(n // 2)].astype(np.int64)


Xm, ym = make_moons()
mx, my = np.meshgrid(np.linspace(-1.6, 2.6, 160), np.linspace(-1.2, 1.7, 160))
grid = torch.tensor(np.c_[mx.ravel(), my.ravel()], dtype=torch.float32)


def make_mlp(hidden=16, depth=2):
    layers, width_in = [], 2
    for _ in range(depth):
        layers += [nn.Linear(width_in, hidden), nn.ReLU()]
        width_in = hidden
    return nn.Sequential(*layers, nn.Linear(width_in, 2))


def train_moons(model, steps, snapshots=()):
    opt = torch.optim.Adam(model.parameters(), lr=0.02)
    X_t, y_t, shots = torch.tensor(Xm), torch.tensor(ym), {}
    for step in range(steps + 1):
        if step in snapshots:
            with torch.no_grad():
                shots[step] = model(grid).softmax(1)[:, 1].reshape(mx.shape).numpy()
        loss = F.cross_entropy(model(X_t), y_t)
        opt.zero_grad(); loss.backward(); opt.step()
    return shots, loss.item()


def draw_moons(ax, prob, title):
    ax.contourf(mx, my, prob, levels=20, cmap="RdBu_r", alpha=0.6, vmin=0, vmax=1)
    ax.contour(mx, my, prob, levels=[0.5], colors="k", linewidths=1.5)
    ax.scatter(*Xm.T, c=ym, cmap="coolwarm", s=8, edgecolor="w", lw=0.3)
    ax.set_title(title); ax.set_xticks([]); ax.set_yticks([])


torch.manual_seed(0)
steps_to_show = [0, 20, 60, 150, 500]
shots, final_loss = train_moons(make_mlp(), max(steps_to_show), steps_to_show)
fig, axes = plt.subplots(1, 5, figsize=(19, 3.6))
for ax, s in zip(axes, steps_to_show):
    draw_moons(ax, shots[s], f"after {s} steps")
plt.suptitle("Figure 6.1 - An MLP (2 hidden layers x 16 ReLU units) learning a curved boundary", fontweight="bold")
plt.tight_layout(); plt.show()

### Interactive: how much capacity does the problem need?
With `depth = 0` the model is a single linear layer, which gives a straight line no matter how long you train. Try 1 hidden unit, then 2, 4, 32.

In [ ]:
def mlp_demo(hidden=16, depth=2, steps=300):
    torch.manual_seed(0)
    model = make_mlp(hidden, depth) if depth > 0 else nn.Linear(2, 2)
    shots, loss = train_moons(model, steps, (steps,))
    fig, ax = plt.subplots(figsize=(6, 4.2))
    draw_moons(ax, shots[steps], f"hidden={hidden}, depth={depth}: loss {loss:.3f}, "
               f"{count_params(model)} params")
    plt.show()


interact(mlp_demo, hidden=widgets.SelectionSlider(options=[1, 2, 4, 8, 16, 32, 64], value=16),
         depth=widgets.IntSlider(value=2, min=0, max=4),
         steps=widgets.IntSlider(value=300, min=50, max=1500, step=50, continuous_update=False))

3Blue1Brown's introduction to neural networks, using handwritten digits:

In [ ]:
youtube("aircAruvnKk")

## 7. Why convolution? Locality and weight sharing

An MLP treats an image as a flat vector: every hidden unit connects to **every pixel**. For images this is wasteful and ignores structure:

| Idea | MLP (fully connected) | Convolution layer |
|---|---|---|
| **Locality**: nearby pixels matter together | each unit sees all pixels | each unit sees a small $k\times k$ patch |
| **Weight sharing**: an edge is an edge anywhere | separate weights per position | the *same* kernel slides over the image |
| **Translation equivariance** | shifting the object changes everything | shifting the object shifts the feature map |
| **Parameters** | grow with image size | independent of image size |

These are the same assumptions you made in Week 3 when you applied one Sobel kernel everywhere. A CNN keeps them but **learns the kernel values**.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.2))
cases = {"Fashion-MNIST 28x28x1": [("FC 784 -> 256", 784 * 256 + 256), ("Conv 5x5, 1 -> 32", 5 * 5 * 32 + 32)],
         "Camera image 224x224x3": [("FC 150,528 -> 1000", 150528 * 1000 + 1000), ("Conv 3x3, 3 -> 64", 3 * 3 * 3 * 64 + 64)]}
for ax, (title, rows) in zip(axes, cases.items()):
    names, vals = zip(*rows)
    bars = ax.barh(names, vals, color=["#d93025", "#188038"])
    ax.set_xscale("log"); ax.set_title(title); ax.set_xlabel("number of learned parameters (log scale)")
    for bar, v in zip(bars, vals):
        ax.text(v * 1.3, bar.get_y() + bar.get_height() / 2, f"{v:,}", va="center")
    ax.set_xlim(10, 1e10)
plt.suptitle("Figure 7.1 - First-layer parameter count: fully connected vs convolution", fontweight="bold")
plt.tight_layout(); plt.show()

In [ ]:
def draw_grid(ax, n=6, highlight=(), color="#fce8b2"):
    for r in range(n):
        for c in range(n):
            ax.add_patch(plt.Rectangle((c, n - 1 - r), 1, 1, fc=color if (r, c) in highlight else "white", ec="#9aa0a6"))
    ax.set_xlim(-0.5, n + 4.5); ax.set_ylim(-1.5, n + 0.5); ax.set_aspect("equal"); ax.axis("off")


fig, axes = plt.subplots(1, 2, figsize=(14, 5))
all_px = [(r, c) for r in range(6) for c in range(6)]
draw_grid(axes[0], highlight=all_px, color="#fad2cf")
for r, c in all_px:
    axes[0].plot([c + 0.5, 9], [5.5 - r, 3], color="#d93025", lw=0.4, alpha=0.6)
axes[0].add_patch(plt.Circle((9.2, 3), 0.35, fc="#fad2cf", ec="#d93025"))
axes[0].set_title("Fully connected: 1 output unit, 36 separate weights")

draw_grid(axes[1], highlight=[(r, c) for r in range(0, 3) for c in range(0, 3)])
for r in range(3, 6):
    for c in range(3, 6):
        axes[1].add_patch(plt.Rectangle((c, 5 - r), 1, 1, fc="#d2e3fc", ec="#9aa0a6"))
for (cy, cx), col in [((1, 1), "#e37400"), ((4, 4), "#1a73e8")]:
    for dr in (-1, 0, 1):
        for dc in (-1, 0, 1):
            axes[1].plot([cx + dc + 0.5, 9], [5.5 - (cy + dr), 5.5 - cy], color=col, lw=0.8)
    axes[1].add_patch(plt.Circle((9.2, 5.5 - cy), 0.35, fc="white", ec=col, lw=2))
axes[1].set_title("Convolution: each output sees a 3x3 patch, and all\noutputs share the SAME 9 weights")
plt.suptitle("Figure 7.2 - Locality and weight sharing", fontweight="bold")
plt.tight_layout(); plt.show()

### A convolution layer *is* `cv2.filter2D`

Here is the key link to Week 3. We create a PyTorch `nn.Conv2d` layer, set its single 3×3 kernel to the Sobel-x kernel
**by hand**, and compare it with OpenCV.

A subtle point: both OpenCV's `filter2D` and PyTorch's `Conv2d` compute **cross-correlation**, which does not flip the kernel.
True mathematical convolution (`scipy.ndimage.convolve`) flips it, and for the antisymmetric Sobel kernel that reverses the sign of every edge.
For *learned* kernels the distinction does not matter, because the network simply learns the flipped kernel.

In [ ]:
cam = skimage.data.camera().astype(np.float32) / 255
sobel_x = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], np.float32)

ref = cv2.filter2D(cam, -1, sobel_x, borderType=cv2.BORDER_CONSTANT)
conv = nn.Conv2d(1, 1, kernel_size=3, padding=1, bias=False)
with torch.no_grad():
    conv.weight[:] = torch.tensor(sobel_x)[None, None]
    out = conv(torch.tensor(cam)[None, None])[0, 0].numpy()
true_conv = ndi.convolve(cam, sobel_x, mode="constant")

print("max |OpenCV filter2D - PyTorch Conv2d| =", np.abs(ref - out).max())
print("max |filter2D + true convolution|     =", np.abs(ref + true_conv).max(), "(same magnitude, opposite sign)")
fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))
for ax, img, title in zip(axes, [ref, out, np.abs(ref - out), true_conv],
                          ["cv2.filter2D (Week 3)", "nn.Conv2d with Sobel weights", "|difference|", "true convolution (flipped)"]):
    im = ax.imshow(img, cmap="seismic" if title != "|difference|" else "gray", vmin=-1 if title != "|difference|" else 0, vmax=1)
    ax.set_title(title); ax.axis("off")
plt.suptitle("Figure 7.3 - A convolution layer with hand-set weights reproduces OpenCV exactly", fontweight="bold")
plt.tight_layout(); plt.show()

3Blue1Brown on what convolution really is, with images and probability:

In [ ]:
youtube("KuXjwB4LzSA")

## 8. CNN building blocks

### 8.1 Convolution layer with many channels
A conv layer maps a $C_\text{in}\times H\times W$ tensor to $C_\text{out}\times H'\times W'$. Each of the $C_\text{out}$ filters
has size $C_\text{in}\times k\times k$ plus a bias:

$$ \#\text{params} = (C_\text{in}\cdot k\cdot k + 1)\cdot C_\text{out} $$

### 8.2 Output size
For input size $n$, kernel $k$, padding $p$, stride $s$ (and dilation $d=1$):

$$ n_\text{out} = \left\lfloor \frac{n + 2p - k}{s} \right\rfloor + 1 $$

"Same" padding for odd $k$ and $s=1$ is $p = (k-1)/2$. A stride of 2 halves the resolution.

In [ ]:
def draw_conv_positions(ax, n, k, p, s):
    size = n + 2 * p
    for r in range(size):
        for c in range(size):
            pad = r < p or c < p or r >= n + p or c >= n + p
            ax.add_patch(plt.Rectangle((c, size - 1 - r), 1, 1, fc="#e8eaed" if pad else "white", ec="#9aa0a6", lw=0.8))
    ax.add_patch(plt.Rectangle((0, size - k), k, k, fill=False, ec="#d93025", lw=2.5))
    n_out = (n + 2 * p - k) // s + 1
    for i in range(n_out):
        for j in range(n_out):
            ax.plot(j * s + k / 2, size - (i * s + k / 2), "o", color="#1a73e8", ms=5)
    ax.set_xlim(-0.2, size + 0.2); ax.set_ylim(-0.2, size + 0.2); ax.set_aspect("equal"); ax.axis("off")
    ax.set_title(f"n={n}, k={k}, p={p}, s={s}  ->  output {n_out}x{n_out}")


fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, cfg in zip(axes, [(5, 3, 0, 1), (5, 3, 1, 1), (7, 3, 1, 2)]):
    draw_conv_positions(ax, *cfg)
plt.suptitle("Figure 8.1 - Padding (grey) and stride. Red: first kernel window. Blue dots: centre of every window = one output pixel",
             fontweight="bold")
plt.tight_layout(); plt.show()

### Interactive: output-size calculator (checked against PyTorch)

In [ ]:
def size_demo(n=28, k=3, p=1, s=1, c_in=1, c_out=32):
    formula = (n + 2 * p - k) // s + 1
    if formula < 1:
        print("Kernel larger than the padded input: no valid output.")
        return
    layer = nn.Conv2d(c_in, c_out, k, stride=s, padding=p)
    real = tuple(layer(torch.zeros(1, c_in, n, n)).shape[1:])
    print(f"formula : floor(({n} + 2*{p} - {k}) / {s}) + 1 = {formula}")
    print(f"PyTorch : output tensor shape {real}")
    print(f"params  : ({c_in}*{k}*{k} + 1) * {c_out} = {count_params(layer):,}")


interact(size_demo, n=widgets.IntSlider(value=28, min=4, max=224), k=widgets.IntSlider(value=3, min=1, max=11, step=2),
         p=widgets.IntSlider(value=1, min=0, max=5), s=widgets.IntSlider(value=1, min=1, max=4),
         c_in=widgets.IntSlider(value=1, min=1, max=64), c_out=widgets.IntSlider(value=32, min=1, max=256))

### 8.3 Pooling
**Max pooling** keeps the strongest response in each window, typically 2×2 with stride 2. It halves the resolution,
adds a little translation invariance and has **no learned parameters**. **Global average pooling** averages each whole
feature map to a single number, turning $C\times H\times W$ into a vector of length $C$ for the classifier.

In [ ]:
fm = np.array([[1, 3, 2, 0], [5, 6, 1, 2], [7, 2, 9, 4], [0, 1, 3, 8]], float)
pooled_max = fm.reshape(2, 2, 2, 2).max((1, 3))
pooled_avg = fm.reshape(2, 2, 2, 2).mean((1, 3))
block_colors = np.array([[0, 0, 1, 1], [0, 0, 1, 1], [2, 2, 3, 3], [2, 2, 3, 3]])
palette = ["#e8f0fe", "#fce8b2", "#ceead6", "#fad2cf"]
fig, axes = plt.subplots(1, 3, figsize=(12, 3.8))
for ax, mat, title, colors in [(axes[0], fm, "4x4 feature map", block_colors),
                               (axes[1], pooled_max, "2x2 max pool", np.array([[0, 1], [2, 3]])),
                               (axes[2], pooled_avg, "2x2 average pool", np.array([[0, 1], [2, 3]]))]:
    n = mat.shape[0]
    o = (4 - n) / 2  # centre the smaller output grids
    for r in range(n):
        for c in range(n):
            ax.add_patch(plt.Rectangle((o + c, o + n - 1 - r), 1, 1, fc=palette[colors[r, c]], ec="k"))
            ax.text(o + c + 0.5, o + n - 0.5 - r, f"{mat[r, c]:g}", ha="center", va="center", fontsize=14)
    ax.set_xlim(0, 4); ax.set_ylim(0, 4); ax.set_aspect("equal"); ax.axis("off"); ax.set_title(title)
plt.suptitle("Figure 8.2 - Pooling summarises each coloured 2x2 block", fontweight="bold")
plt.tight_layout(); plt.show()

### 8.4 Receptive field: how much of the image does a unit see?
Each 3×3 conv adds 2 pixels to the receptive field, and every stride-2 pooling **doubles** the growth of later layers.
Deep units therefore see large regions and can respond to whole object parts. That is the learned counterpart of the
**image pyramids** from Week 5. The recurrence, with $j$ the cumulative stride ("jump"), is:

$$ r_\ell = r_{\ell-1} + (k_\ell - 1)\, j_{\ell-1}, \qquad j_\ell = j_{\ell-1}\, s_\ell $$

In [ ]:
def receptive_fields(layers):
    r, j, out = 1, 1, []
    for k, s in layers:
        r, j = r + (k - 1) * j, j * s
        out.append(r)
    return out


plain = [(3, 1)] * 8
with_pool = [(3, 1), (3, 1), (2, 2), (3, 1), (3, 1), (2, 2), (3, 1), (3, 1)]
rf_plain, rf_pool = receptive_fields(plain), receptive_fields(with_pool)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(range(1, 9), rf_plain, "o-", lw=2, label="8 x conv 3x3")
axes[0].plot(range(1, 9), rf_pool, "s-", lw=2, label="conv, conv, pool, conv, conv, pool, ...")
axes[0].set_xlabel("layer"); axes[0].set_ylabel("receptive field (pixels)"); axes[0].grid(alpha=0.3); axes[0].legend()
axes[0].set_title("Receptive field grows with depth; pooling accelerates it")
axes[1].imshow(cam, cmap="gray")
cy, cx = 250, 260
for layer, color in [(1, "#fbbc04"), (4, "#34a853"), (8, "#ea4335")]:
    rf = rf_pool[layer - 1] * 4  # scale x4: the layers act on a 4x downsampled copy of this image
    axes[1].add_patch(plt.Rectangle((cx - rf / 2, cy - rf / 2), rf, rf, fill=False, ec=color, lw=2.5,
                                    label=f"layer {layer}: {rf_pool[layer - 1]} px (x4)"))
axes[1].legend(loc="lower right"); axes[1].axis("off")
axes[1].set_title("What one unit 'sees' at increasing depth\n(boxes drawn x4, as if the network ran on a 4x smaller copy)")
plt.suptitle("Figure 8.3 - Receptive field", fontweight="bold")
plt.tight_layout(); plt.show()

## 9. Build and train a CNN

Our network has three convolution layers (two followed by max pooling). The final 64×7×7 feature maps are flattened and fed to
one linear layer. The first layer uses 5×5 kernels so that the learned filters are large enough to inspect in Section 10.
With about 88k parameters it is **2.7× smaller** than the MLP we compare it with.

In [ ]:
class FashionCNN(nn.Module):
    def __init__(self, width=32, n_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, width, 5, padding=2), nn.ReLU(), nn.MaxPool2d(2),              # 28x28 -> 14x14
            nn.Conv2d(width, 2 * width, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),      # 14x14 -> 7x7
            nn.Conv2d(2 * width, 2 * width, 3, padding=1), nn.ReLU(),                   # 7x7
            nn.Flatten())                                                               # -> 2*width*7*7 numbers
        self.classifier = nn.Linear(2 * width * 7 * 7, n_classes)

    def forward(self, x):
        return self.classifier(self.features(x))


def make_fashion_mlp():
    return nn.Sequential(nn.Flatten(), nn.Linear(784, 256), nn.ReLU(),
                         nn.Linear(256, 128), nn.ReLU(), nn.Linear(128, 10))


layer_table(FashionCNN(), (1, 28, 28))

In [ ]:
# Figure 9.1: the tensor shapes flowing through the network
shapes = [("input", 1, 28), ("conv5x5\n+ReLU", 32, 28), ("maxpool", 32, 14), ("conv3x3\n+ReLU", 64, 14),
          ("maxpool", 64, 7), ("conv3x3\n+ReLU", 64, 7), ("flatten", 3136, 1), ("linear", 10, 1)]
fig, ax = plt.subplots(figsize=(17, 4.2))
x = 0
for name, ch, hw in shapes:
    h = 0.4 + 2.6 * hw / 28
    w = 0.25 + 0.9 * np.log2(ch + 1) / 7
    for off in (0.12, 0.06, 0):
        ax.add_patch(plt.Rectangle((x + off, 0.5 - h / 2 + off), w, h, fc="#d2e3fc" if hw > 1 else "#ceead6",
                                   ec="#1a73e8", lw=1))
    ax.text(x + w / 2, -1.25, f"{name}\n{ch}x{hw}x{hw}", ha="center", va="top", fontsize=9)
    x += w + 1.0
ax.set_xlim(-0.3, x); ax.set_ylim(-2.6, 2.2); ax.axis("off")
ax.set_title("Figure 9.1 - Resolution shrinks while the number of channels (feature types) grows", fontweight="bold")
plt.show()

### The training loop, line by line
Every PyTorch training loop has the same five steps: **forward → loss → zero gradients → backward → step**.
We normalise pixels to zero mean and unit variance (Fashion-MNIST mean 0.286, std 0.353), then train with the
**Adam** optimiser (Kingma & Ba, 2015), a gradient-descent variant that adapts the step size of every weight.

> We evaluate on the official test set after every epoch only to *watch* learning. In Session B you will learn why a
> separate **validation** set is needed for model selection.

In [ ]:
def to_tensor(X):
    return (torch.tensor(X, dtype=torch.float32).unsqueeze(1) / 255 - 0.286) / 0.353


train_loader = DataLoader(TensorDataset(to_tensor(Xtr), torch.tensor(ytr)), batch_size=BATCH, shuffle=True)
test_loader = DataLoader(TensorDataset(to_tensor(Xte), torch.tensor(yte)), batch_size=512)


def train(model, epochs=EPOCHS, lr=1e-3):
    model = model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    hist = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    for epoch in range(1, epochs + 1):
        model.train()
        t0, run_loss, run_correct, seen = time.time(), 0.0, 0, 0
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            logits = model(x)            # 1. forward pass: scores for 10 classes
            loss = loss_fn(logits, y)    # 2. cross-entropy: how wrong are we?
            opt.zero_grad()              # 3. clear gradients from the previous step
            loss.backward()              # 4. backpropagation: dLoss/dw for every weight
            opt.step()                   # 5. gradient-descent update
            run_loss += loss.item() * len(y)
            run_correct += (logits.argmax(1) == y).sum().item()
            seen += len(y)
        model.eval()
        with torch.no_grad():
            val_loss, val_correct = 0.0, 0
            for x, y in test_loader:
                x, y = x.to(DEVICE), y.to(DEVICE)
                logits = model(x)
                val_loss += loss_fn(logits, y).item() * len(y)
                val_correct += (logits.argmax(1) == y).sum().item()
        for key, v in [("train_loss", run_loss / seen), ("train_acc", run_correct / seen),
                       ("val_loss", val_loss / len(yte)), ("val_acc", val_correct / len(yte))]:
            hist[key].append(v)
        print(f"epoch {epoch}/{epochs}  train loss {hist['train_loss'][-1]:.3f}  "
              f"test acc {hist['val_acc'][-1]:.3f}  ({time.time() - t0:.1f}s)")
    return model, hist


set_seed(0)
t0 = time.time()
mlp, h_mlp = train(make_fashion_mlp())
t_mlp = time.time() - t0
t0 = time.time()
cnn, h_cnn = train(FashionCNN())
t_cnn = time.time() - t0
print(f"\nMLP: {count_params(mlp):,} params, {t_mlp:.0f}s | CNN: {count_params(cnn):,} params, {t_cnn:.0f}s on {DEVICE}")

In [ ]:
plot_history({"MLP": h_mlp, "CNN": h_cnn}, title="Figure 9.2 - Training curves: MLP vs CNN (dashed = train, solid = test)")

## 10. Looking inside: what did the CNN learn?

### 10.1 First-layer filters vs the kernels you designed
**Figure 10.1** compares the hand-designed kernels from Weeks 3–5 (top) with the 32 first-layer kernels learned by the CNN (bottom;
red = positive weight, blue = negative). Look for oriented edge detectors (like Sobel), blob or centre-surround detectors (like the
Laplacian) and stripe detectors (like Gabor). Nobody told the network to build these; they emerged because they are useful for the task.
After only a few epochs (or on a CPU run) many kernels still look noisy. Train longer and they become smoother and more structured.

In [ ]:
hand = {"Sobel x": sobel_x, "Sobel y": sobel_x.T, "Laplacian": np.array([[0, 1, 0], [1, -4, 1], [0, 1, 0]], float)}
for theta in (0, 45, 90, 135):
    hand[f"Gabor {theta} deg"] = np.real(gabor_kernel(0.25, theta=np.deg2rad(theta), sigma_x=2, sigma_y=2))
fig, axes = plt.subplots(1, len(hand), figsize=(15, 2.4))
for ax, (name, k) in zip(axes, hand.items()):
    m = np.abs(k).max()
    ax.imshow(k, cmap="seismic", vmin=-m, vmax=m); ax.set_title(name, fontsize=9); ax.axis("off")
plt.suptitle("Hand-designed kernels (Weeks 3-5)", fontweight="bold"); plt.tight_layout(); plt.show()

learned = cnn.features[0].weight.detach().cpu().numpy()[:, 0]
fig, axes = plt.subplots(4, 8, figsize=(12, 6.3))
for ax, k in zip(axes.ravel(), learned):
    m = np.abs(k).max()
    ax.imshow(k, cmap="seismic", vmin=-m, vmax=m); ax.axis("off")
plt.suptitle("Figure 10.1 - The 32 kernels your CNN learned in layer 1 (5x5)", fontweight="bold")
plt.tight_layout(); plt.show()

### 10.2 Feature maps
The response of each filter to one image. Early layers respond to edges and simple strokes. Deeper layers respond to larger, more abstract patterns at lower resolution.

In [ ]:
sample = to_tensor(Xte[:1]).to(DEVICE)
maps, h = {}, sample
cnn.eval()
with torch.no_grad():
    for i, layer in enumerate(cnn.features):
        h = layer(h)
        if i in (1, 4, 7):
            maps[{1: "layer 1 (28x28)", 4: "layer 2 (14x14)", 7: "layer 3 (7x7)"}[i]] = h[0].cpu().numpy()
fig, axes = plt.subplots(3, 9, figsize=(16, 6))
for r, (name, fmaps) in enumerate(maps.items()):
    axes[r, 0].imshow(Xte[0], cmap="gray_r"); axes[r, 0].set_title(CLASSES[yte[0]] if r == 0 else "")
    axes[r, 0].set_ylabel(name)
    top = np.argsort(fmaps.mean((1, 2)))[::-1][:8]
    for j, ch in enumerate(top):
        axes[r, j + 1].imshow(fmaps[ch], cmap="viridis"); axes[r, j + 1].set_title(f"ch {ch}", fontsize=8)
for ax in axes.ravel():
    ax.set_xticks([]); ax.set_yticks([])
plt.suptitle("Figure 10.2 - The 8 most active feature maps at each depth", fontweight="bold")
plt.tight_layout(); plt.show()

### 10.3 Embeddings: raw pixels vs HOG vs learned features
We project 2,000 test images to 2D with **PCA** (principal component analysis) using three representations: raw pixels,
HOG, and the 3,136-number vector the CNN produces before its classifier. The title also reports the **1-nearest-neighbour accuracy**
in the full feature space, which measures how well each representation groups images of the same class.

In [ ]:
def pca2(F_):
    F_ = F_ - F_.mean(0)
    _, _, Vt = np.linalg.svd(F_, full_matrices=False)
    return F_ @ Vt[:2].T


def nn1_accuracy(F_, y, n=1000):
    F_ = F_ / (np.linalg.norm(F_, axis=1, keepdims=True) + 1e-8)
    a, b = F_[:n], F_[n:2 * n]
    return (y[:n][(b @ a.T).argmax(1)] == y[n:2 * n]).mean()  # label of the nearest image in set a


n_emb = min(2000, len(Xte))
with torch.no_grad():
    cnn_feat = torch.cat([cnn.features(to_tensor(Xte[i:i + 500]).to(DEVICE)).cpu()
                          for i in range(0, n_emb, 500)]).numpy()
reps = {"Raw pixels (784-D)": Xte[:n_emb].reshape(n_emb, -1).astype(np.float32),
        "HOG (324-D)": H_te[:n_emb], "CNN features (3136-D)": cnn_feat}
fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))
for ax, (name, F_) in zip(axes, reps.items()):
    P = pca2(F_)
    sc = ax.scatter(P[:, 0], P[:, 1], c=yte[:n_emb], cmap="tab10", s=5)
    ax.set_title(f"{name}\n1-NN accuracy: {nn1_accuracy(F_, yte[:n_emb], n_emb // 2):.3f}")
    ax.set_xticks([]); ax.set_yticks([])
handles = [plt.Line2D([], [], marker="o", ls="", color=plt.cm.tab10(i / 9), label=c) for i, c in enumerate(CLASSES)]
fig.legend(handles=handles, loc="center right")
plt.suptitle("Figure 10.3 - Learned features group the classes much more cleanly", fontweight="bold")
plt.tight_layout(rect=(0, 0, 0.9, 1)); plt.show()

### Interactive: explore the CNN's predictions
Pick any test image and see the network's probability for every class. The true class is shown in green.

In [ ]:
def predict_demo(index=0):
    with torch.no_grad():
        probs = cnn(to_tensor(Xte[index:index + 1]).to(DEVICE)).softmax(1)[0].cpu().numpy()
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), gridspec_kw={"width_ratios": [1, 2.5]})
    axes[0].imshow(Xte[index], cmap="gray_r"); axes[0].axis("off")
    axes[0].set_title(f"true: {CLASSES[yte[index]]}")
    colors = ["#188038" if c == yte[index] else "#9aa0a6" for c in range(10)]
    axes[1].barh(CLASSES, probs, color=colors); axes[1].set_xlim(0, 1); axes[1].invert_yaxis()
    axes[1].set_title(f"predicted: {CLASSES[probs.argmax()]} ({probs.max():.2f})")
    plt.tight_layout(); plt.show()


interact(predict_demo, index=widgets.IntSlider(value=0, min=0, max=len(Xte) - 1, continuous_update=False))

## 11. Results and error analysis

In [ ]:
def accuracy_and_preds(model):
    model.eval()
    with torch.no_grad():
        preds = torch.cat([model(x.to(DEVICE)).argmax(1).cpu() for x, _ in test_loader]).numpy()
    return (preds == yte).mean(), preds


acc_mlp, pred_mlp = accuracy_and_preds(mlp)
acc_cnn, pred_cnn = accuracy_and_preds(cnn)
RESULTS["MLP (raw pixels)"] = (acc_mlp, count_params(mlp))
RESULTS["CNN"] = (acc_cnn, count_params(cnn))

names = list(RESULTS)
accs = [RESULTS[n][0] for n in names]
fig, ax = plt.subplots(figsize=(10, 4.2))
bars = ax.bar(names, accs, color=["#9aa0a6", "#fbbc04", "#8ab4f8", "#188038"])
for bar, n in zip(bars, names):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
            f"{RESULTS[n][0]:.3f}\n{RESULTS[n][1]:,} params", ha="center", fontsize=9)
ax.set_ylim(0, 1.1); ax.set_ylabel("test accuracy"); ax.grid(axis="y", alpha=0.3)
ax.set_title("Figure 11.1 - Hand-crafted vs learned: test accuracy on Fashion-MNIST", fontweight="bold")
plt.show()

In [ ]:
plot_confusion(confusion_matrix(yte, pred_cnn, 10), CLASSES, title="Figure 11.2 - CNN confusion matrix (rows sum to 1)")

per_class_hog = [(pred_hog[yte == c] == c).mean() for c in range(10)]
per_class_cnn = [(pred_cnn[yte == c] == c).mean() for c in range(10)]
xs = np.arange(10)
fig, ax = plt.subplots(figsize=(13, 4))
ax.bar(xs - 0.2, per_class_hog, 0.4, label="HOG + linear", color="#fbbc04")
ax.bar(xs + 0.2, per_class_cnn, 0.4, label="CNN", color="#188038")
ax.set_xticks(xs, CLASSES, rotation=20); ax.set_ylim(0, 1.05); ax.legend(); ax.grid(axis="y", alpha=0.3)
ax.set_title("Figure 11.3 - Per-class accuracy: where does learning help most?", fontweight="bold")
plt.show()

In [ ]:
wrong = np.where(pred_cnn != yte)[0][:16]
fig, axes = plt.subplots(2, 8, figsize=(16, 4.6))
for ax, i in zip(axes.ravel(), wrong):
    ax.imshow(Xte[i], cmap="gray_r"); ax.axis("off")
    ax.set_title(f"true {CLASSES[yte[i]]}\npred {CLASSES[pred_cnn[i]]}", fontsize=8, color="#d93025")
plt.suptitle("Figure 11.4 - CNN mistakes: many are ambiguous even for humans", fontweight="bold")
plt.tight_layout(); plt.show()

**Discussion.** With the full training settings (GPU: 60k images, 8 epochs; CPU: 20k images, 3 epochs) the CNN beats both HOG
and the MLP while using 2.7× fewer parameters than the MLP, because convolution builds in the right assumptions (locality, weight
sharing). If your run shows otherwise, check the number of epochs printed at the top. In the very short CI/"smoke" setting the CNN
has not converged yet. Its remaining errors are concentrated in *Shirt / T-shirt / Pullover / Coat*: at 28×28 pixels
some of these are genuinely ambiguous. In industry this is where you would improve the **data** (higher resolution, better
lighting, clearer class definitions) rather than the model. That is the machine-vision engineer's job.

## 12. Exercises (with self-checks)

Replace each `raise NotImplementedError` with your code, then run the check cell below it.
✅ means your answer behaves correctly. The checks test *behaviour*, so they do not reveal the answer.

### E1. Convolution output size
Implement the output-size formula from Section 8.2, including **dilation** $d$. A dilated kernel covers $d(k-1)+1$ pixels.

In [ ]:
def conv_output_size(n, k, p=0, s=1, d=1):
    """Output height/width of a convolution: input n, kernel k, padding p, stride s, dilation d."""
    # TODO: your code here
    raise NotImplementedError

In [ ]:
def _test_e1():
    for n, k, p, s, d in [(28, 5, 2, 1, 1), (28, 3, 0, 1, 1), (224, 7, 3, 2, 1),
                          (64, 3, 1, 2, 1), (32, 3, 2, 1, 2), (15, 4, 1, 3, 1)]:
        want = nn.Conv2d(1, 1, k, stride=s, padding=p, dilation=d)(torch.zeros(1, 1, n, n)).shape[-1]
        got = conv_output_size(n, k, p, s, d)
        assert got == want, f"n={n}, k={k}, p={p}, s={s}, d={d}: PyTorch gives {want}, you gave {got}"


check("E1 conv_output_size", _test_e1)

### E2. A hand-built Laplacian layer
Return an `nn.Conv2d` layer that applies the 4-neighbour Laplacian kernel below to a 1-channel image, keeps the image size,
has **no bias**, and is **frozen** (its weights must not be updated by training). Freezing is a key tool in Session B.

$$ K = \begin{bmatrix} 0 & 1 & 0 \\ 1 & -4 & 1 \\ 0 & 1 & 0 \end{bmatrix} $$

In [ ]:
LAPLACIAN = np.array([[0, 1, 0], [1, -4, 1], [0, 1, 0]], np.float32)


def laplacian_layer():
    # TODO: create the layer, copy LAPLACIAN into its weights, freeze it
    raise NotImplementedError

In [ ]:
def _test_e2():
    layer = laplacian_layer()
    assert isinstance(layer, nn.Conv2d), "return an nn.Conv2d"
    assert layer.bias is None, "the layer should have no bias (bias=False)"
    assert not any(p.requires_grad for p in layer.parameters()), "freeze the weights (requires_grad = False)"
    with torch.no_grad():
        out = layer(torch.tensor(cam)[None, None])[0, 0].numpy()
    assert out.shape == cam.shape, f"output shape {out.shape} should equal input shape {cam.shape}: check padding"
    ref = cv2.filter2D(cam, -1, LAPLACIAN, borderType=cv2.BORDER_CONSTANT)
    assert np.allclose(out, ref, atol=1e-5), "output differs from cv2.filter2D with the Laplacian kernel"


check("E2 laplacian_layer", _test_e2)

### E3. Counting parameters
Implement the parameter-count formula for a conv layer with or without bias. Then use it (on paper) to check the `Params`
column of the table printed in Section 9.

In [ ]:
def conv_params(c_in, c_out, k, bias=True):
    # TODO: your code here
    raise NotImplementedError

In [ ]:
def _test_e3():
    for c_in, c_out, k, bias in [(1, 32, 5, True), (32, 64, 3, True), (3, 64, 7, False), (64, 128, 1, True)]:
        want = count_params(nn.Conv2d(c_in, c_out, k, bias=bias))
        got = conv_params(c_in, c_out, k, bias)
        assert got == want, f"c_in={c_in}, c_out={c_out}, k={k}, bias={bias}: expected {want:,}, got {got}"


check("E3 conv_params", _test_e3)

### E4. Experiment (no self-check)
Make **one** change to `FashionCNN`: halve or double `width`, remove the second `MaxPool2d`, or replace `ReLU` with `nn.Tanh()`.
Train it for 2 epochs with `train(...)` and record test accuracy, parameter count and training time in a small table.
Explain the result in 2–3 sentences using ideas from this notebook (capacity, receptive field, vanishing gradients).

In [ ]:
# E4: your experiment here, e.g.
# variant, h_variant = train(FashionCNN(width=16), epochs=2)

## 13. Challenge (independent)

**Parameter-efficient inspection model.** Edge devices (Week 9) have tight memory and compute budgets.
Design a CNN with **at most 20,000 parameters** that reaches **at least 88% test accuracy** on Fashion-MNIST.

1. Try at least three architectures and plot **accuracy vs parameter count** (log x-axis). The best designs sit on the upper-left *Pareto front*.
2. Measure inference time per image on your laptop GPU and on the CPU (`torch.device("cpu")`).
3. Hint: depthwise-separable convolutions (`groups=` in `nn.Conv2d`, as in MobileNet) and global average pooling save many parameters.

In [ ]:
# Your challenge code here

## 14. Quick check (auto-marked)

In [ ]:
quiz([
    {
        "q": 'A conv layer has 16 input channels, 32 output channels and 3x3 kernels with bias. How many parameters does it have?',
        "options": [
            '288',
            '4,608',
            '4,640',
            '147,456',
        ],
        "answer": '4e98eea1bf',
        "explain": '(16 x 3 x 3 + 1) x 32 = 4,640: each of the 32 filters spans all 16 input channels and has one bias.',
    },
    {
        "q": 'Input 64x64, kernel 5x5, padding 0, stride 1. What is the output size?',
        "options": [
            '64x64',
            '60x60',
            '59x59',
            '30x30',
        ],
        "answer": 'dee5ec4730',
        "explain": 'floor((64 + 0 - 5) / 1) + 1 = 60.',
    },
    {
        "q": 'Why is ReLU usually preferred over sigmoid in deep networks?',
        "options": [
            'It is bounded between 0 and 1',
            'Its gradient does not vanish for positive inputs, so deep networks train faster',
            'It makes the network linear',
            'It has learnable parameters',
        ],
        "answer": '97eee5af34',
        "explain": 'Sigmoid saturates (gradient near 0 for large |z|); ReLU passes a gradient of 1 for all positive inputs.',
    },
    {
        "q": 'What does torch.nn.Conv2d actually compute?',
        "options": [
            'True convolution with a flipped kernel',
            'Cross-correlation, the same as cv2.filter2D',
            'A 2D Fourier transform',
            'Max pooling',
        ],
        "answer": 'e4f983a310',
        "explain": 'Neither PyTorch nor filter2D flips the kernel. For learned kernels the difference is irrelevant.',
    },
    {
        "q": 'Your loss suddenly jumps to NaN after a few steps. What is the most likely cause?',
        "options": [
            'Learning rate too large',
            'Learning rate too small',
            'Too many test images',
            'Using ReLU',
        ],
        "answer": '2b1229ae1b',
        "explain": 'Steps that are too large overshoot the minimum and the loss diverges (Figure 5.1, red path).',
    },
    {
        "q": 'What is the main advantage of learned features over hand-crafted ones?',
        "options": [
            'They need no data',
            'They are always easier to interpret',
            'They are optimised from data for the task, capturing variation a designer did not anticipate',
            'They are always faster to compute',
        ],
        "answer": '67fac7543d',
        "explain": "Hand-crafted features encode the designer's assumptions. Learned features adapt to the real variation in the data.",
    },
    {
        "q": 'What does weight sharing in a convolution layer mean?',
        "options": [
            'All layers use the same kernel',
            'The same kernel is applied at every image position',
            'Weights are copied from a pretrained network',
            'Kernels are shared between the MLP and CNN',
        ],
        "answer": '46d42b3d14',
        "explain": 'One set of k x k weights slides over the whole image, like Sobel in Week 3, which makes the parameter count independent of image size.',
    },
], title='Week 06A quiz')

## 15. Exit ticket (reflection)
Answer in 2–3 sentences each:
1. Name one inspection task where you would still choose a **hand-crafted** pipeline over a CNN, and justify it.
2. Your CNN confuses *Shirt* and *T-shirt*. Give two **data-side** fixes and one **model-side** fix.
3. In your own words, explain why a CNN needs far fewer parameters than an MLP on images.

## 16. What's new (2025–26)

> - **Foundation backbones.** Large self-supervised models such as Meta's **DINOv3** (2025) produce general-purpose features
>   that transfer to many inspection tasks with little labelled data (Week 6B, Week 11).
> - **CNNs are not obsolete.** Modern convolutional designs (ConvNeXt, and the backbones inside YOLO11/YOLO26 in Week 8) remain
>   the standard for real-time and embedded vision, where their locality bias makes them data-efficient and fast.
> - **Tooling.** PyTorch 2.x `torch.compile` speeds up training with one line, and the `timm` library offers over a thousand pretrained image models.

## 17. References and further exploration

**Seminal papers**
- Rumelhart, Hinton & Williams (1986). *Learning representations by back-propagating errors.* Nature 323. [doi:10.1038/323533a0](https://doi.org/10.1038/323533a0)
- LeCun, Bottou, Bengio & Haffner (1998). *Gradient-based learning applied to document recognition.* Proc. IEEE 86(11). [doi:10.1109/5.726791](https://doi.org/10.1109/5.726791)
- Dalal & Triggs (2005). *Histograms of Oriented Gradients for Human Detection.* CVPR. [doi:10.1109/CVPR.2005.177](https://doi.org/10.1109/CVPR.2005.177)
- Krizhevsky, Sutskever & Hinton (2012). *ImageNet Classification with Deep Convolutional Neural Networks* (AlexNet). NeurIPS. [paper](https://papers.nips.cc/paper/2012/hash/c399862d3b9d6b76c8436e924a68c45b-Abstract.html)
- Zeiler & Fergus (2014). *Visualizing and Understanding Convolutional Networks.* ECCV. [arXiv:1311.2901](https://arxiv.org/abs/1311.2901)
- Kingma & Ba (2015). *Adam: A Method for Stochastic Optimization.* ICLR. [arXiv:1412.6980](https://arxiv.org/abs/1412.6980)
- Xiao, Rasul & Vollgraf (2017). *Fashion-MNIST.* [arXiv:1708.07747](https://arxiv.org/abs/1708.07747)

**Textbooks (free online)**
- Prince, S. J. D. (2023). *Understanding Deep Learning.* MIT Press, ch. 3–4 (networks), 6–7 (training, backprop), 10 (CNNs). [udlbook.github.io](https://udlbook.github.io/udlbook/)
- Szeliski, R. (2022). *Computer Vision: Algorithms and Applications*, 2nd ed., ch. 5 (Deep learning). [szeliski.org/Book](https://szeliski.org/Book/)
- Torralba, Isola & Freeman (2024). *Foundations of Computer Vision.* MIT Press. [visionbook.mit.edu](https://visionbook.mit.edu/)
- Goodfellow, Bengio & Courville (2016). *Deep Learning*, ch. 6 and 9. [deeplearningbook.org](https://www.deeplearningbook.org/)

**Courses, videos and interactive demos**
- Stanford CS231n: *Deep Learning for Computer Vision*, course notes. [cs231n.github.io](https://cs231n.github.io/)
- **CNN Explainer** (Georgia Tech Polo Club): an interactive CNN running in your browser. [poloclub.github.io/cnn-explainer](https://poloclub.github.io/cnn-explainer/)
- **TensorFlow Playground**: train small networks on 2D data interactively. [playground.tensorflow.org](https://playground.tensorflow.org/)
- 3Blue1Brown: *Neural networks* video series. [youtube.com/@3blue1brown](https://www.youtube.com/playlist?list=PLZHQObOWTQDNU6R1_67000Dx_ZCJB-3pi)
- PyTorch tutorial: *Learn the Basics*. [docs.pytorch.org/tutorials](https://docs.pytorch.org/tutorials/beginner/basics/intro.html)

**Data:** Fashion-MNIST © Zalando SE, MIT licence. `skimage.data.camera` from scikit-image (public domain).

## 18. Key takeaways
- Hand-crafted features encode a designer's assumptions. **Learned features** are optimised from data and handle variation the designer did not foresee.
- A neuron is $\sigma(\mathbf{w}^\top\mathbf{x}+b)$. Networks learn by **gradient descent** on a loss, with gradients from **backpropagation**.
- A conv layer is the Week 3 `filter2D` operation with **learned kernels**. Locality and weight sharing make it efficient for images.
- Output size $\lfloor (n+2p-k)/s \rfloor + 1$; params $(C_\text{in}k^2+1)C_\text{out}$; receptive field grows with depth and pooling.
- A small CNN beats HOG + linear and the MLP. Its first-layer filters resemble the Sobel/Gabor kernels you designed by hand.

| Term | Meaning |
|---|---|
| Epoch | one pass over the training set |
| Batch | the examples used for one gradient step |
| Learning rate | the step size $\eta$ in gradient descent |
| Logits | raw class scores before softmax |
| Feature map | the output of one filter over the image |
| Receptive field | the input region that influences one unit |

**Next session (6B):** training *well* on real photos: data splits, augmentation, overfitting, transfer learning from ImageNet, and Grad-CAM explanations.